In [1]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from matplotlib.colors import to_hex
import matplotlib.pyplot as plt
import networkx as nx
import math
import base64
import io
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import transform
import numpy as np

Cria um mapa do Brasil, que mostra o contorno dos estados, em base64

In [2]:

ESCALA = 60                              # o mesmo valor usado nas posições dos vértices
COS = math.cos(math.radians(-15))        # o mesmo fator de correção da longitude

def projeta(lon, lat, z=None):
    # idêntica à função posicao() do mapa
    return lon * COS * ESCALA, -lat * ESCALA

# Contorno dos estados (use geobr.read_country() se quiser só o contorno do país)
URL = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"
estados = gpd.read_file(URL)
geoms = [transform(projeta, g) for g in estados.geometry]

xmin = min(g.bounds[0] for g in geoms)
ymin = min(g.bounds[1] for g in geoms)   # menor y = norte (y é invertido)
xmax = max(g.bounds[2] for g in geoms)
ymax = max(g.bounds[3] for g in geoms)
w, h = xmax - xmin, ymax - ymin

# Figura sem margens, ocupando exatamente a extensão dos dados
px = 8000
fig = plt.figure(figsize=(px / 100, px * h / w / 100), dpi=100)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymax, ymin)                  # invertido, como no pyvis
ax.axis("off")

cmap = plt.cm.Greens
cores_estados = cmap(np.linspace(0.35, 0.5, len(geoms)))

for i, g in enumerate(geoms):
    partes = g.geoms if g.geom_type == "MultiPolygon" else [g]
    cor = cores_estados[i % len(cores_estados)]

    for p in partes:
        x, y = p.exterior.xy
        ax.fill(
            x, y,
            facecolor=cor,
            edgecolor="#051425",
            linewidth=0.6
        )
buf = io.BytesIO()
fig.savefig(buf, format="png", transparent=True)
plt.close(fig)
img_b64 = base64.b64encode(buf.getvalue()).decode()

C:\Users\Diogo\AppData\Local\Temp\ipykernel_5776\2259608501.py:11: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  geoms = [transform(projeta, g) for g in estados.geometry]


Modela a base de dados como um grafo direcionado

In [ ]:
df = pd.read_csv("../dados_processados/origem_destino_customedio.csv", sep=";", index_col=0)


df = df[df["origem"] == "SBMG"]
df = df[df["mes"] == 6]

G = nx.from_pandas_edgelist(
    df,
    source="origem",
    target="destino",
    edge_attr=["custo_por_assento", "empresa"],
    create_using=nx.MultiDiGraph,
)


# Nota do Diogo:

Por enquanto, o mapa do Brasil está recebendo a modelagem de todo o nosso conjunto de dados. Porém, ele deverá receber apenas um subgrafo que contém os caminhos mais baratos. É aqui que aplicaremos yen também!

In [4]:
aero = pd.read_csv("../dados_processados/locais_aerodromos.csv", sep=";", index_col=0)
coords = aero.set_index("ICAO")[["Latitude", "Longitude"]].to_dict("index")
nomes = aero.set_index("ICAO")["Nome"].to_dict()

# 2) Cores por companhia (G é o MultiDiGraph construído antes)
companhias = sorted({d["empresa"] for _, _, d in G.edges(data=True)})
cor_cias = {
    'ABJ': '#009B3A',  
    'AZU': '#0038A8',  
    'CQB': '#E30613',  
    'GLO': '#FF7020',  
    'TAM': '#ED1650'   
}

# 3) Escala: graus -> pixels
ESCALA = 60  # aumente para espalhar mais o mapa

def posicao(icao):
    lat, lon = coords[icao]["Latitude"], coords[icao]["Longitude"]
    # correção simples de projeção: longitude encolhe com cos(latitude)
    import math
    x = lon * math.cos(math.radians(-15)) * ESCALA   # -15° ~ latitude média do Brasil
    y = -lat * ESCALA                                # inverte: no pyvis y cresce para baixo
    return x, y

# 4) Rede pyvis sem física, com vértices fixos
net = Network(height="900px", width="100%", directed=True, bgcolor="#5959cc")
net.toggle_physics(False)

sem_coord = []
for no in G.nodes:
    if no not in coords:
        sem_coord.append(no)
        continue
    x, y = posicao(no)
    net.add_node(
        no,
        label=no,
        title=f"{no} - {nomes[no]}",
        x=x, y=y,
        physics=False,
        size=12,
        color="#4a90d9",
    )

# 5) Arestas (arestas paralelas ganham curvatura diferente)
for u, v, k, d in G.edges(keys=True, data=True):
    if u in sem_coord or v in sem_coord:
        continue
    net.add_edge(
        u, v,
        title=f"{d['empresa']}: R$ {d['custo_por_assento']:.2f}",
        color=cor_cias[d["empresa"]],
        value=d["custo_por_assento"],   # espessura proporcional ao custo
        smooth={"type": "curvedCW", "roundness": 0.1 + 0.1 * k},
    )

if sem_coord:
    print("Aeroportos sem coordenadas (ignorados):", sem_coord)

net.write_html("mapa_rotas.html")

script = f"""
<script>
  const fundo = new Image();
  fundo.src = "data:image/png;base64,{img_b64}";
  network.on("beforeDrawing", function (ctx) {{
      if (fundo.complete) {{
          ctx.drawImage(fundo, {xmin}, {ymin}, {w}, {h});
      }}
  }});
  fundo.onload = function () {{ network.redraw(); }};
</script>
"""

with open("mapa_rotas.html", "r", encoding="utf-8") as f:
    html = f.read()

html = html.replace("</body>", script + "</body>")

with open("mapa_rotas.html", "w", encoding="utf-8") as f:
    f.write(html)

Aeroportos sem coordenadas (ignorados): ['SDLO', 'SNCL', 'SBSI', 'SBVC', 'SBPO', 'SBFC', 'SBJE', 'SSOU', 'SBTC', 'SNCP', 'SBAC', 'SSIM']
